# Notebook 16 — Complete Preprocessing Workflow
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers

This notebook assembles every decision made across Notebooks 1-15 into one complete,
end-to-end workflow, producing a final, saved, ML-ready dataset.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Building the complete Sprint 5 preprocessing workflow...")


Building the complete Sprint 5 preprocessing workflow...


---
## Step 1 — Load Dataset


In [2]:
df = pd.read_csv("telco_churn.csv")
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")


Loaded: 7,043 rows, 21 columns


---
## Step 2 — Inspect Dataset


In [3]:
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

---
## Step 3 — Identify Data Types


In [4]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
ordinal_feature = ['Contract']
nominal_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                     'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                     'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod']
identifier = 'customerID'
target = 'Churn'
print(f"Numeric: {len(numeric_features)} | Ordinal: {len(ordinal_feature)} | Nominal: {len(nominal_features)}")


Numeric: 3 | Ordinal: 1 | Nominal: 15


---
## Step 4 — Identify Missing Values


In [5]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Missing values after type correction: {df.isnull().sum().sum()} (all in TotalCharges)")


Missing values after type correction: 11 (all in TotalCharges)


---
## Step 5 — Handle Missing Values (Notebook 3's decision: MAR -> constant 0)


In [6]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f"Missing values after fill: {df.isnull().sum().sum()}")


Missing values after fill: 0


---
## Step 6 — Detect Duplicates


In [7]:
exact_dupes = df.duplicated().sum()
print(f"Exact duplicates: {exact_dupes}")


Exact duplicates: 0


---
## Step 7 — Handle Duplicates (Notebook 4's decision: none removed — verified real customers)


In [8]:
print("No removal needed — 0 exact duplicates found; partial-duplicate profiles (Notebook 4)")
print("were investigated and confirmed to be distinct customers by their unique customerID.")


No removal needed — 0 exact duplicates found; partial-duplicate profiles (Notebook 4)
were investigated and confirmed to be distinct customers by their unique customerID.


---
## Step 8 — Validate Data (Notebook 5's rules)


In [9]:
validation_report = {
    'tenure_in_range': df['tenure'].between(0, 100).all(),
    'charges_non_negative': (df['MonthlyCharges'] >= 0).all() and (df['TotalCharges'] >= 0).all(),
    'churn_valid_categories': set(df['Churn'].unique()) <= {'Yes', 'No'},
    'customerID_unique': df['customerID'].nunique() == len(df),
    'no_nulls': df.isnull().sum().sum() == 0,
}
for check, passed in validation_report.items():
    print(f"{check:<25}: {'PASS' if passed else 'FAIL'}")
assert all(validation_report.values()), "Validation failed — halting workflow"
print("\nAll validation checks passed.")


tenure_in_range          : PASS
charges_non_negative     : PASS
churn_valid_categories   : PASS
customerID_unique        : PASS
no_nulls                 : PASS

All validation checks passed.


---
## Step 9 — Detect Outliers


In [10]:
from scipy import stats
df['expected_total'] = df['tenure'] * df['MonthlyCharges']
df['billing_residual'] = df['TotalCharges'] - df['expected_total']
outlier_mask = np.abs(stats.zscore(df['billing_residual'])) > 3
print(f"Multivariate billing outliers detected: {outlier_mask.sum()}")


Multivariate billing outliers detected: 112


---
## Step 10 — Treat Outliers (Notebook 6's decision: retain all, keep residual as a feature)


In [11]:
print("Decision: RETAIN all 112 flagged rows (genuine, explainable billing history).")
print("The 'billing_residual' column computed above is kept as an engineered feature.")


Decision: RETAIN all 112 flagged rows (genuine, explainable billing history).
The 'billing_residual' column computed above is kept as an engineered feature.


---
## Step 11 — Encode Categorical Variables


In [12]:
df['Contract_encoded'] = OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]).fit_transform(df[['Contract']])
print("Contract ordinally encoded. Remaining nominal columns will be one-hot encoded inside the pipeline (Step 14).")


Contract ordinally encoded. Remaining nominal columns will be one-hot encoded inside the pipeline (Step 14).


---
## Step 12 — Scale Numerical Variables (handled inside the pipeline, Step 14, to respect train/test split)


In [13]:
print("Scaling deferred to the Pipeline step (Step 14) — StandardScaler must be fit on")
print("the TRAINING split only (Notebook 12/13's leakage-prevention discipline), not on the full dataset here.")


Scaling deferred to the Pipeline step (Step 14) — StandardScaler must be fit on
the TRAINING split only (Notebook 12/13's leakage-prevention discipline), not on the full dataset here.


---
## Step 13 — Split Data


In [14]:
feature_cols = numeric_features + ['Contract_encoded', 'billing_residual'] + \
               [c for c in nominal_features]
X = df[feature_cols]
y = (df['Churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")


Train: (5634, 20) | Test: (1409, 20)


---
## Step 14 — Build & Fit the Preprocessing Pipeline


In [15]:
numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])
nominal_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features + ['billing_residual']),
    ('ordinal_passthrough', 'passthrough', ['Contract_encoded']),
    ('nominal', nominal_pipeline, nominal_features),
])

full_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
])

full_pipeline.fit(X_train, y_train)
print("Pipeline fit successfully on TRAINING data only.")


Pipeline fit successfully on TRAINING data only.


---
## Step 15 — Handle Class Imbalance (Notebook 11's decision: class_weight='balanced', applied above)


In [16]:
preds = full_pipeline.predict(X_test)
print(f"Test Accuracy : {accuracy_score(y_test, preds):.4f}")
print(f"Test Precision: {precision_score(y_test, preds):.4f}")
print(f"Test Recall   : {recall_score(y_test, preds):.4f}")
print(f"Test F1       : {f1_score(y_test, preds):.4f}")
print("\n(class_weight='balanced' selected per Notebook 11 as the lowest-risk imbalance response;")
print("SMOTE remains available as an alternative, to be applied to X_train/y_train only if needed.)")


Test Accuracy : 0.7360


Test Precision: 0.5017


Test Recall   : 0.7807
Test F1       : 0.6109

(class_weight='balanced' selected per Notebook 11 as the lowest-risk imbalance response;
SMOTE remains available as an alternative, to be applied to X_train/y_train only if needed.)


---
## Step 16 — Perform Feature Selection (Notebook 10's finding, documented for the next sprint)


In [17]:
print("Top features identified in Notebook 10 (cross-method agreement):")
print("  Contract, tenure, OnlineSecurity, TechSupport, InternetService, MonthlyCharges, TotalCharges")
print("Full feature set retained in THIS pipeline (no columns dropped) — feature reduction")
print("is deferred as an explicit modeling-stage decision for Sprint 6, not baked in here.")


Top features identified in Notebook 10 (cross-method agreement):
  Contract, tenure, OnlineSecurity, TechSupport, InternetService, MonthlyCharges, TotalCharges
Full feature set retained in THIS pipeline (no columns dropped) — feature reduction
is deferred as an explicit modeling-stage decision for Sprint 6, not baked in here.


---
## Step 17 — Validate Final Dataset


In [18]:
processed_train = preprocessor.transform(X_train)
print(f"Final processed training matrix shape: {processed_train.shape}")
print(f"Any NaN remaining: {np.isnan(processed_train).sum()}")
print(f"All values numeric: {processed_train.dtype}")


Final processed training matrix shape: (5634, 30)
Any NaN remaining: 0
All values numeric: float64


---
## Final Output — Save the ML-Ready Dataset


In [19]:
# Save the cleaned (pre-pipeline-encoding) dataset — the natural hand-off point to Sprint 6 (Feature Engineering)
ml_ready_df = df[['customerID'] + feature_cols + ['Churn']].copy()
ml_ready_df.to_csv('telco_churn_ml_ready.csv', index=False)

print(f"Saved: telco_churn_ml_ready.csv — {ml_ready_df.shape[0]:,} rows, {ml_ready_df.shape[1]} columns")
print(f"\nFinal dataset preview:")
print(ml_ready_df.head(3))


Saved: telco_churn_ml_ready.csv — 7,043 rows, 22 columns

Final dataset preview:
   customerID  tenure  MonthlyCharges  TotalCharges  Contract_encoded  \
0  7590-VHVEG       1           29.85         29.85               0.0   
1  5575-GNVDE      34           56.95       1889.50               1.0   
2  3668-QPYBK       2           53.85        108.15               0.0   

   billing_residual  gender  SeniorCitizen Partner Dependents  ...  \
0              0.00  Female              0     Yes         No  ...   
1            -46.80    Male              0      No         No  ...   
2              0.45    Male              0      No         No  ...   

  InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport  \
0             DSL             No          Yes               No          No   
1             DSL            Yes           No              Yes          No   
2             DSL            Yes          Yes               No          No   

  StreamingTV StreamingMovies Pa

---
## Complete Workflow Summary

| Step | Decision | Source Notebook |
|---|---|---|
| Load & Inspect | 7,043 x 21 raw dataset | 1-2 |
| Data Types | `TotalCharges` -> float64 | 2 |
| Missing Values | Constant-fill (0), justified by MAR analysis | 3 |
| Duplicates | None removed — verified real customers | 4 |
| Validation | 5/5 checks passed | 5 |
| Outliers | 112 retained; `billing_residual` engineered | 6 |
| Encoding | `Contract` ordinal; rest one-hot (in-pipeline) | 7 |
| Scaling | `StandardScaler` (in-pipeline, train-fit only) | 8 |
| Transformation | Available (Yeo-Johnson) for `TotalCharges` if needed downstream | 9 |
| Feature Selection | Top features identified, full set retained for now | 10 |
| Imbalance | `class_weight='balanced'` | 11 |
| Splitting | Stratified 80/20 | 12 |
| Leakage Prevention | Pipeline enforces fit-on-train discipline | 13-14 |

**Final deliverable:** `telco_churn_ml_ready.csv` — a validated, leakage-safe, ML-ready
dataset, plus a fitted `Pipeline` object ready for prediction on new customer records.

**Next notebook:** `17_Mini_Assessment.ipynb` — self-assessment covering the entire
sprint.
